# 贵州茅台「董秘」LoRA 微调（第三课作业 B）

基座：Qwen3.5-2B（原生多模态，文本任务）　方法：4bit QLoRA，r=16

数据：404 组「投资者问 / 董秘答（带出处）」，由大模型从 11 份财报原文生成后抽检。

在 AutoDL 镜像上实测：3 轮 5.8 分钟、6 轮 11.0 分钟（RTX 4080S 32G）。

In [ ]:
!pip install -q -U torch==2.6.0 torchvision==0.21.0
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl
import torch, transformers
print(torch.__version__, torch.cuda.is_available(), transformers.__version__)

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # 国内下模型走镜像
os.environ['HF_HOME'] = '/root/autodl-tmp/hf'         # 别塞系统盘

## 1. 数据长什么样

整段渲染好的结构化文本（Qwen 对话模板），训练时把 `text` 字段原样喂进去。

In [ ]:
import json
rows = [json.loads(l) for l in open('sft_train.jsonl', encoding='utf-8')]
print(len(rows), '条；train/val =',
      sum(r['split']=='train' for r in rows), '/', sum(r['split']=='val' for r in rows))
print(rows[0]['text'][:600])

## 2. 训练（4bit QLoRA）

可训练参数应为 **10,911,744** —— 和课件录屏里的数字一致。

In [ ]:
!python train_lora_dongmi.py --backend peft --epochs 6 --rank 16 \
    --batch 2 --grad-accum 4 --out lora_dongmi

## 3. 四格对照评测

160 条冻结题库（60 条训练过 + 20 条从未训练，各含原问法和换问法），
上下文由本机预先检索好放进 `eval_with_context.jsonl`，这里只做生成。

In [ ]:
!python eval_dongmi.py --configs finetuned,kb_finetuned --adapter lora_dongmi \
    --out eval_results.jsonl --batch 6 --max-new 200

In [ ]:
import json
rows = [json.loads(l) for l in open('eval_results.jsonl', encoding='utf-8')]
for r in rows[:2]:
    print('问：', r['question'])
    print('答：', r['output'][:300], '\n')

## 4. 结论

微调学会的是**格式与口吻**（带出处率 44/60 → 60/60），事实仍然要靠知识库递进来
（没材料 8% 命中 → 给材料 43% → 微调后再给材料 55%）。

完整四格表见 `B_四格对照.md`，分析见 `B_一页结论.md`。